<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/d3-cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 3 · Notebook 5 — Convolutional networks on CIFAR-10

On Tuesday you designed kernels by hand. On Wednesday a network learned its
own numbers, but it flattened each image into a list and threw away which
pixel is next to which. Today's networks keep the picture: their first layers
are **kernels**, like Tuesday's, with numbers learned the way Wednesday's
were. That is a **convolutional neural network** (CNN).

The data is CIFAR-10: 60,000 tiny colour photos (32×32) in ten classes. This
afternoon's competition uses the same data, so everything here is practice
for it.

Same rules: 🔮 predict, ✏️ replace the `...`, ✅ check, 🐛 debug. Swap driver
and navigator at every `##`.

In [ ]:
import os
import hashlib
import pathlib
import shutil
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torchvision
from torchvision import transforms


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


def fetch_from_mirror(url, dest, sha256):
    """Download url to dest and check its sha256. Returns True if the file is good."""
    # The original data servers get slow when a whole class downloads at once,
    # so we keep a copy of each file on our own fast server.
    dest = pathlib.Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    def good(path):
        with open(path, "rb") as f:
            return hashlib.file_digest(f, "sha256").hexdigest() == sha256

    if dest.exists() and good(dest):
        return True
    # Download to a .part file first, so a stopped download never looks finished.
    part = dest.with_name(dest.name + ".part")
    try:
        # Our server turns away requests that look like they come from a robot,
        # so we give ours a name.
        request = urllib.request.Request(url, headers={"User-Agent": "tk-accelerator-notebooks"})
        with urllib.request.urlopen(request, timeout=30) as reply, open(part, "wb") as out:
            shutil.copyfileobj(reply, out)
        if not good(part):
            raise ValueError("the checksum does not match")
        part.replace(dest)
        return True
    except Exception:
        part.unlink(missing_ok=True)
        print("Our copy of the data did not download, so we are using the original source.")
        return False


class DataMissing(Exception):
    """Stops a cell with one plain message instead of a long error."""

    def __init__(self, message):
        super().__init__(message)
        self.message = message

    def _render_traceback_(self):
        return [self.message]


def require_data(name, *paths):
    """Stop with a plain message unless the downloaded files are on disk."""
    if not all(pathlib.Path(p).exists() for p in paths):
        raise DataMissing(f"The {name} data isn't downloaded yet. Run the ⬇️ Download cell above first.")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# The instructors' checker sets NB_CHECK to run this notebook quickly; ignore it.
QUICK = os.environ.get("NB_CHECK") == "1"
torch.manual_seed(0)
print("Running on", device)

## 1. What a convolution layer does to the shape

`nn.Conv2d(in_channels, out_channels, kernel_size)` is a set of Tuesday's
kernels: `out_channels` of them, each one `kernel_size` × `kernel_size` and
as deep as the input has channels.

In [ ]:
x = torch.zeros(1, 3, 32, 32)   # 1 image, 3 colour channels, 32 × 32 pixels
conv = nn.Conv2d(3, 16, kernel_size=3)
print("input :", tuple(x.shape))
print("output:", tuple(conv(x).shape))

🔮 **Predict.** Why did 32 become 30? (Think about Tuesday's `convolve` and the
edge pixels.) What will the output be with `padding=1`, which adds a border
of zeros? And with `kernel_size=5` and no padding?

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
for kwargs in [dict(kernel_size=3, padding=1), dict(kernel_size=5), dict(kernel_size=3, stride=2)]:
    print(kwargs, "->", tuple(nn.Conv2d(3, 16, **kwargs)(x).shape))
print("MaxPool2d(2) on 30×30 ->", tuple(nn.MaxPool2d(2)(torch.zeros(1, 16, 30, 30)).shape))

✏️ **Exercise 1a.** Write `out_size(n, kernel, padding=0, stride=1)`: the
output width for an input `n` pixels wide. The recipe: add the padding on
both sides (`n + 2 * padding`), subtract the kernel size, divide by the
stride with `//`, then add one. Check it against the numbers printed above
before running the check.

In [ ]:
def out_size(n, kernel, padding=0, stride=1):
    return ...

In [ ]:
check(is_done(out_size(32, 3)), "out_size still returns ... .")
check(out_size(32, 3) == 30, "32 wide, kernel 3, no padding: 30.")
check(out_size(32, 3, padding=1) == 32, "Padding 1 on each side keeps 32.")
check(out_size(32, 5) == 28, "Kernel 5 loses 4 pixels.")
check(out_size(32, 3, stride=2) == 15, "Stride 2 on 32 with kernel 3 gives 15.")
check(out_size(30, 2, stride=2) == 15, "A 2×2 max pool with stride 2 halves 30 to 15.")

## 2. Build a CNN, and fix one

A CNN usually stacks: convolution, ReLU, pooling (shrink), repeated, then
**flatten** and a normal `Linear` layer to give the ten class scores.

🐛 **Debugging challenge.** This model crashes. Run the cell, read the last line
of the error, and fix the number that is wrong. (Use `out_size` to work out the
shape after each layer, or print it.)

In [ ]:
def make_cnn():
    return nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32 -> 16
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16 -> 8
        nn.Flatten(),
        nn.Linear(32 * 16 * 16, 64), nn.ReLU(),
        nn.Linear(64, 10),
    )


model = make_cnn()
print(tuple(model(torch.zeros(1, 3, 32, 32)).shape))

In [ ]:
check(tuple(make_cnn()(torch.zeros(2, 3, 32, 32)).shape) == (2, 10), "The model should turn 2 images into 2 × 10 scores.")

Everyone continues with this working model (the code is hidden so it doesn't
spoil the challenge; double-click to see it afterwards).

In [ ]:
#@title Run this to continue { display-mode: "form" }
def make_cnn():
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(2048, 64), nn.ReLU(),
        nn.Linear(64, 10),
    )


print("numbers to learn:", sum(p.numel() for p in make_cnn().parameters()))

🔮 **Predict.** Wednesday's digits network had about 2,400 numbers. This one has
more than 100,000. Which layer do you think holds most of them? Check by
running the next cell.

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
for name, p in make_cnn().named_parameters():
    print(f"{name:10s} {str(tuple(p.shape)):18s} {p.numel():>8,}")

## 3. Loading CIFAR-10

The data is about 170 MB, and the next two cells get it: first a download cell, then a loading cell. A
**transform** turns each photo into a tensor and scales its values; an
**augmentation** changes training photos a little at random, so the model
never sees exactly the same picture twice.

⬇️ **Download the data (170 MB).** Run the next cell when you start section 3. It downloads 170 MB, so skip it if you are only reading. It is safe to run twice. "Run all" runs it too.

In [ ]:
# ⬇️ Download cell: CIFAR-10 (170 MB)
import tarfile

if pathlib.Path("data/cifar-10-batches-py").exists():
    print("CIFAR-10 is already downloaded. Nothing to do.")
else:
    if fetch_from_mirror("https://data.tk.sg/cifar-10/cifar-10-python.tar.gz", "data/cifar-10-python.tar.gz",
                         "6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce"):
        # Unpack into a spare folder first, so a stopped unpack never looks finished.
        shutil.rmtree("data/cifar-unpacking", ignore_errors=True)
        with tarfile.open("data/cifar-10-python.tar.gz") as tar:
            tar.extractall("data/cifar-unpacking", filter="data")
        pathlib.Path("data/cifar-unpacking/cifar-10-batches-py").rename("data/cifar-10-batches-py")
        shutil.rmtree("data/cifar-unpacking")
    else:
        torchvision.datasets.CIFAR10("data", download=True)
    print("Done: CIFAR-10 downloaded (170 MB).")

In [ ]:
CLASSES = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
normalise = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
train_tf = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.ToTensor(), normalise])
plain_tf = transforms.Compose([transforms.ToTensor(), normalise])

require_data("CIFAR-10", "data/cifar-10-batches-py")
full_train = torchvision.datasets.CIFAR10("data", train=True, download=False, transform=train_tf)
full_val = torchvision.datasets.CIFAR10("data", train=True, download=False, transform=plain_tf)
print(len(full_train), "training photos")

🔮 **Predict.** Why is a horizontal flip a safe augmentation for these photos?
Name one augmentation that would be a bad idea for CIFAR-10, and one that would
be fine for histology slides but not for photos.

In [ ]:
my_prediction = ""  # @param {type:"string"}

The last 5,000 training photos become the **validation** set (the prelims).
The official test set is kept for the competition (the O-level).

In [ ]:
order = torch.randperm(50000, generator=torch.Generator().manual_seed(0))
if QUICK:
    n_train = 1000
elif device.type == "cuda":
    n_train = 45000
else:
    n_train = 10000   # CPU: a subset keeps each epoch short
train_set = torch.utils.data.Subset(full_train, order[:n_train].tolist())
val_set = torch.utils.data.Subset(full_val, order[45000:45000 + (500 if QUICK else 5000)].tolist())
print(f"train on {len(train_set)} photos, validate on {len(val_set)}")

✏️ **Exercise 3a.** Make `train_loader`, a `torch.utils.data.DataLoader` over
`train_set` with `batch_size=64` and `shuffle=True`. Then make `val_loader`
over `val_set` with `batch_size=256` and no shuffling.

In [ ]:
train_loader = ...
val_loader = ...

In [ ]:
check(is_done(train_loader, val_loader), "Replace both ... .")
xb, yb = next(iter(train_loader))
check(tuple(xb.shape) == (64, 3, 32, 32), f"A training batch should be 64 × 3 × 32 × 32, but it is {tuple(xb.shape)}.")
check(train_loader.batch_size == 64 and val_loader.batch_size == 256, "Check both batch sizes.")
check(isinstance(train_loader.sampler, torch.utils.data.RandomSampler), "Shuffle the training data.")
check(isinstance(val_loader.sampler, torch.utils.data.SequentialSampler), "Don't shuffle validation.")

Everyone continues with these loaders.

In [ ]:
#@title Run this to continue { display-mode: "form" }
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=256)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow((images[i].permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1))
    ax.set_title(CLASSES[labels[i]])
    ax.axis("off")
plt.show()

## 4. Train and evaluate

This is Wednesday's loop with two additions: `.to(device)` moves data to the
GPU if there is one, and `model.train()` / `model.eval()` tell layers that act
differently while training (you will meet some in the competition) which mode
they are in.

In [ ]:
def run_epoch(model, optimizer, loss_fn):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()


def predict_all(model, loader):
    model.eval()
    preds, truth = [], []
    with torch.no_grad():
        for images, labels in loader:
            preds.append(model(images.to(device)).argmax(dim=1).cpu())
            truth.append(labels)
    return torch.cat(preds), torch.cat(truth)

🔮 **Predict.** Ten classes, so guessing gives 10%. What validation accuracy do
you expect after 3 epochs: 20%, 50% or 80%?

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
torch.manual_seed(0)
model = make_cnn().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(1 if QUICK else 3):
    run_epoch(model, optimizer, loss_fn)
    preds, truth = predict_all(model, val_loader)
    print(f"epoch {epoch + 1}: validation accuracy {(preds == truth).float().mean():.1%}")

A single accuracy number hides *which* mistakes the model makes. A **confusion
matrix** counts them: row = the true class, column = the model's answer.

🔮 **Predict.** Which two classes do you think the model mixes up most?

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
confusion = np.zeros((10, 10), dtype=int)
for t, p in zip(truth.tolist(), preds.tolist()):
    confusion[t, p] += 1

plt.figure(figsize=(6, 6))
plt.imshow(confusion, cmap="Blues")
plt.xticks(range(10), CLASSES, rotation=45)
plt.yticks(range(10), CLASSES)
plt.xlabel("model's answer")
plt.ylabel("true class")
for i in range(10):
    for j in range(10):
        plt.text(j, i, confusion[i, j], ha="center", va="center", fontsize=7)
plt.show()

✏️ **Exercise 4a.** Find the most common mistake: the largest number in
`confusion` that is **not** on the diagonal. Make `true_class` and
`guessed_class` its row and column. (Hint: copy the matrix, set the diagonal
to 0 with `np.fill_diagonal`, then `np.unravel_index(m.argmax(), m.shape)`
turns the position of the largest value into a row and a column.)

In [ ]:
true_class, guessed_class = ..., ...

In [ ]:
check(is_done(true_class, guessed_class), "Replace both ... .")
check(true_class != guessed_class, "A mistake is off the diagonal: the two classes must differ.")
off = confusion.copy()
np.fill_diagonal(off, -1)
check(confusion[true_class, guessed_class] == off.max(), "That cell is not the largest mistake. Look again.")
print(f"Most common mistake: a {CLASSES[true_class]} called a {CLASSES[guessed_class]} "
      f"({confusion[true_class, guessed_class]} times).")

## 5. What did the first layer learn?

The first layer has 16 kernels, each 3×3 and 3 colours deep. Tuesday you
designed kernels; here are the ones the network found.

In [ ]:
weights = model[0].weight.detach().cpu()
weights = (weights - weights.min()) / (weights.max() - weights.min())
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(weights[i].permute(1, 2, 0))
    ax.axis("off")
plt.suptitle("the 16 learned first-layer kernels")
plt.show()

🔮 **Spot them.** Can you see any that look like Tuesday's edge kernels (light
on one side, dark on the other)? Any that respond to one colour? Write down
the numbers (counting from 0) of two kernels and what you think each one
detects. After only a few epochs they are noisy; models trained for longer
show cleaner edge and colour detectors.

In [ ]:
my_kernel_notes = ""  # @param {type:"string"}

That is the whole idea of a CNN: Tuesday's kernels, Wednesday's learning.
This afternoon you will build your own and compete.

## Extension (if you finish early)

Open the Day 3 extension track in Sidecar, or:

- Add a third convolution block. What must change in the `Linear` layer?
- Add `nn.BatchNorm2d(16)` after the first convolution. Change only that and
  compare validation accuracy.
- Train without `RandomHorizontalFlip`. Is validation accuracy better or worse
  after the same number of epochs?

In [ ]:
# Your extension work here.